# Phân tích baseline v5
Đọc output v5 ordered_tool_plan_caller. Planner chọn danh sách tool có thứ tự một lần; caller điền args/values cho vị trí hiện tại. Code giữ thứ tự và tự dừng khi hết plan. `done` nghĩa hết danh sách, chưa chứng minh hoàn thành đủ request. `score_proxy` là metric xấp xỉ.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path("/kaggle/working/lifemate_baseline_v5")  # đổi thành thư mục report đã tải
if not ROOT.exists():
    ROOT = Path("lifemate_baseline_v5")

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.exists() else []

def read_run(mode):
    base = ROOT / mode
    return {
        "summary": json.loads((base/"summary.json").read_text()) if (base/"summary.json").exists() else {},
        "cases": pd.DataFrame(jsonl(base/"traces.jsonl")),
        "metrics": pd.read_csv(base/"cases.csv") if (base/"cases.csv").exists() and (base/"cases.csv").stat().st_size else pd.DataFrame(),
        "status": json.loads((base/"run_status.json").read_text()) if (base/"run_status.json").exists() else {},
    }
runs = {mode: read_run(mode) for mode in ("workflow_gold", "baseline_asr")}
for mode, run in runs.items():
    print(mode, run["status"])
    if run["summary"] and run["summary"].get("architecture") != "ordered_tool_plan_caller":
        print("Report dùng kiến trúc khác/phiên bản v5 cũ:", run["summary"].get("architecture"))
    display(run["summary"])


## Metric, lỗi và điểm dừng
`caller_blocked`: caller không thể xử lý bước hiện tại, dừng không sửa plan. `skip` là bỏ một slot, không gọi env; cần kiểm nhánh có thực sự không áp dụng. `invalid_model_output`: vi phạm contract. `validation_failed`: env.validate từ chối. `tool_failed`: env.call thất bại nghiệp vụ, kể cả ok=true/done=false. Những lỗi này dừng ngay, không retry/replan. `environment_no_progress`: read lặp; `duplicate_write`: code dừng trước write trùng đã thành công.


In [ ]:
for mode, run in runs.items():
    df = run["metrics"]
    if df.empty:
        continue
    print(mode)
    display(df[[c for c in ["id", "score_proxy", "state_hash_match", "stop_reason", "tool_count", "selected_tool_count", "plan_step_count", "processed_steps", "skipped_steps", "tool_set_recall", "tool_order_recall", "tool_plan_exact_match", "planner_calls", "caller_calls", "llm_calls", "agent_latency_s"] if c in df]])
    display(df.stop_reason.value_counts().rename("count").to_frame())
    display(df.loc[df.score_proxy < 1, ["id", "error_flags", "missing_tool_names", "extra_tool_names"]])


## Chi phí hai node
Trace chứa node_outputs cho `plan` và `call`. Planner luôn một lượt; caller có thể nhiều lượt để thực thi nhiều bước. Plan rỗng không gọi caller. N slot xử lý hết cần 1+N lượt Qwen, không có lượt done riêng. Trace ghi tool_plan, step_index, selected_tool, caller_output và action thực thi. Batch gom request của nhiều mẫu, mỗi request giữ prompt/schema riêng.


In [ ]:
node_rows = []
for mode, run in runs.items():
    for case in run["cases"].to_dict("records"):
        for event in case.get("trace", []):
            for node, output in event.get("node_outputs", {}).items():
                node_rows.append({"mode": mode, "id": case["id"], "decision": event["decision"], "node": node,
                    **{k: output.get(k) for k in ["prompt_tokens", "generation_tokens", "latency_s"]}})
nodes = pd.DataFrame(node_rows)
if not nodes.empty:
    display(nodes.groupby(["mode", "node"])[["prompt_tokens", "generation_tokens", "latency_s"]].agg(["count", "mean", "max"]))


In [ ]:
def show_case(sample_id, mode="workflow_gold"):
    run = runs[mode]
    traces = run["cases"]
    if traces.empty or sample_id not in set(traces.id):
        print("Không tìm thấy", sample_id)
        return
    metric = run["metrics"]
    if not metric.empty:
        display(metric.loc[metric.id == sample_id].T)
    case = traces.loc[traces.id == sample_id].iloc[0].to_dict()
    print(json.dumps(case, ensure_ascii=False, indent=2))

# show_case("<sample_id>")


## Diễn giải
- `tool_set_recall` đo độ bao phủ các tên tool trong nhãn train; không phải metric hoàn thành task. Recall thấp là dấu hiệu planner chọn thiếu tool.
- Caller chỉ nhìn schema của selected_tool tại step_index; code gắn tool và tăng con trỏ. Tên lặp trong tool_plan là nhiều slot riêng.
- tool_order_recall là LCS tên tool có tính lặp / số gold calls nghiệp vụ, chỉ diagnostic offline. tool_set_recall không kiểm thứ tự.
- Kiểm skip có căn cứ và processed_steps (gồm call thành công + skip); step_results không thay cho observation thật.
- Plan sai/thiếu không được bổ sung runtime. Số đối tượng chỉ biết sau lookup có thể vượt số slot đã lập.
- Đối chiếu request với tool calls để phát hiện done sớm, thiếu action hoặc chọn sai tool/field/value.
- `validation.ok=true` không xác minh ID hoặc nội dung đúng ý user. Khi tool trả done=false, v5 dừng mẫu; response được giữ trong trace.
- `local_diagnostic` là log của code, không gửi model dưới dạng feedback. Không có reflector hoặc vòng fine-tune tự động.
- So tốc độ bằng experiment_wall_s/samples_per_second; workers>1 có thời gian chờ batch. Dùng workers=1 để đo latency riêng.
